# Importing the Data
### CelebA Dataset 

We will be utilizing the CelebA dataset for this exploration \
Pixels values will be normalized from [0,255] -> [-1, 1] \
This is done to work in conjunction with the Generator's tanh activation output \

In [16]:
import torchvision.transforms as transforms

# mean and std of channels
mean = [0.5, 0.5, 0.5] # [R,G,B]
std = [0.5, 0.5, 0.5] # [R,G,B]

transform = transforms.Compose([
    transforms.CenterCrop((178, 178)),
    transforms.Resize((64, 64)),
    transforms.ToTensor(), # pixel values go from [0,255] to [0,1]
    transforms.Normalize(
        mean=mean,
        std=std
    ) # pixel values go from [0,1] to [-1,1]
])

In [17]:
from torchvision.datasets import CelebA
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_dataset = CelebA(root='data', split='train', transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)


Downloading...
From (original): https://drive.google.com/uc?id=0B7EVK8r0v71pZjFTYXZWM3FlRnM
From (redirected): https://drive.usercontent.google.com/download?id=0B7EVK8r0v71pZjFTYXZWM3FlRnM&confirm=t&uuid=2b67b82c-f470-422d-aa33-6a0fc3f265f4
To: /Users/maahirpaliwal/github_repos/Generative-Modelling/vanilla-GAN/data/celeba/img_align_celeba.zip
100%|██████████| 1.44G/1.44G [03:08<00:00, 7.67MB/s]
Downloading...
From: https://drive.google.com/uc?id=0B7EVK8r0v71pblRyaVFSWGxPY0U
To: /Users/maahirpaliwal/github_repos/Generative-Modelling/vanilla-GAN/data/celeba/list_attr_celeba.txt
100%|██████████| 26.7M/26.7M [00:03<00:00, 7.40MB/s]
Downloading...
From: https://drive.google.com/uc?id=1_ee_0u7vcNLOfNLegJRHmolfH5ICW-XS
To: /Users/maahirpaliwal/github_repos/Generative-Modelling/vanilla-GAN/data/celeba/identity_CelebA.txt
100%|██████████| 3.42M/3.42M [00:00<00:00, 4.85MB/s]
Downloading...
From: https://drive.google.com/uc?id=0B7EVK8r0v71pbThiMVRxWXZ4dU0
To: /Users/maahirpaliwal/github_repos/Gen

# Defining the Model

### Generator : 
Input: a latent z of dimension noise_dim (assume gaussian distribution) \
Output : A proposed image

### Discriminator
Input: a proposed image, namely the output G(z) \
Output : A binary classification label 
* '1' : The discriminator predicts that the image comes from distribution pdata (image is part of training set) 
* '0' : The dicrimimator predicts that the image comes from distribution pG (image was generated by Generator) 

### Goal 
The minimax game has achieved optimality when: 

$pG(x) = pData(x)$ (Optimal generator for optimal D) \
$DG(x) = \frac{pdata(x)}{pdata(x) + pG(x)} = 0.5$ (Discriminator is guessing)

In [ ]:
import torch.nn as nn 

class Generator(nn.Module):
    def __init__(self, latent_dim=100, img_height=64, img_width=64, img_channels=3):
        super().__init__()
        self.latent_dim = latent_dim
        self.model = nn.Sequential(
            nn.Linear(self.latent_dim, 256), 
            nn.BatchNorm1d(256),
            nn.ReLU(), 

            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),

            # Flatten to image dimensions
            # Tanh b/c pixel values are normalized between -1 and 1
            nn.Linear(512, img_height * img_width * img_channels),
            nn.Tanh(), 

            # [B, H * W * C] -> [B, C, H, W]
            nn.Unflatten(-1, (img_channels, img_height, img_width))
        )

    def forward(self, z):
        x = self.model(z)
        return x

In [ ]:
import torch.nn as nn

class Discriminator(nn.Module):
    def __init__(self, img_height=64, img_width=64, img_channels=3):
        super().__init__()

        # P(Y=1|x) = D(x) = sigmoid(f(x))
        # we will use BCEWithLogitsLoss below, because it is more numerically stable than using a plain Sigmoid followed by a BCELoss
        self.model = nn.Sequential(
            nn.Flatten(start_dim=1), 

            nn.Linear(img_height * img_width * img_channels, 512), 
            nn.BatchNorm1d(512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 1),
        )

    def forward(self, x):
        y = self.model(x)
        return y

In [ ]:
import torch

LATENT_DIM = 100
NUM_EPOCHS = 50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
generator = Generator(latent_dim=LATENT_DIM)
discriminator = Discriminator()

generator_optimizer = torch.optim.AdamW(generator.parameters())
discriminator_optimizer = torch.optim.AdamW(discriminator.parameters())

In [ ]:
import torchvision.transforms as transforms

# mean and std of channels
mean = [0.5, 0.5, 0.5] # [R,G,B]
std = [0.5, 0.5, 0.5] # [R,G,B]

transform = transforms.Compose([
    transforms.CenterCrop((178, 178)),
    transforms.Resize((64, 64)),
    transforms.ToTensor(), # pixel values go from [0,255] to [0,1]
    transforms.Normalize() # pixel values go from [0,1] to [-1,1]
])

In [ ]:
def train_gan(generator,
              discriminator,
              generator_optimizer, 
              discriminator_optimizer, 
              train_loader, 
              device, 
              latent_dim=100, 
              num_epochs=50, 
              loss_function=nn.BCEWithLogitsLoss(), 
              ):
    for epoch in range(num_epochs):
        for i, data in enumerate(train_loader):
            real_images, _ = data
            real_images = real_images.to(device)

            # ------  train discriminator on real data ------
            discriminator_optimizer.zero_grad()                        # zero out the gradients from the previous batch
            real_labels = torch.ones(real_images(0), 1, device=device) # (B, 1) -> '1' for each image in the batch
            real_outputs = discriminator(real_images)
            real_loss = loss_function(real_labels, real_outputs)
            real_loss.backward()                                       # computes weight updates for real images

            # ------ train discriminator on fake data ------
            z = torch.randn(real_images.size(0), latent_dim, device=device)
            fake_images = generator(z)
            fake_labels = torch.zeros(real_images(0), 1, device=device) # (B, 1) -> '0' for every img in the batch
            fake_outputs = discriminator(fake_images.detach())          # detach does not propagate gradients back through G
            fake_loss = loss_function(fake_labels, fake_outputs)
            fake_loss.backward()
            discriminator_optimizer.step()

            # ------ train the generator on the real data ------
            generator_optimizer.zero_grad()
            fake_labels = torch.ones(real_images(0), 1, device=device)  # # (B, 1) -> '1' for each image in the batch
            fake_outputs = discriminator(fake_images)                   # D(G(z))
            gen_loss = loss_function(fake_outputs, fake_labels)         # BCE(D(G(z)), 1) : B/C generator ideally wants to fool discriminator
            gen_loss.backward()
            generator_optimizer.step()

            if i % 100 == 0:
                print(f"Epoch [{epoch+1} / {num_epochs}], Step [{i + 1} / {len(train_loader)}] ,"
                      f"Discriminator Loss : {real_loss.item() + fake_loss.ite():.4f}"
                      f"Generator Loss : {gen_loss.item():.4f}")





